`MACHINE LEARNING | ANALÍA ELIZABETH GOMORY | ARGENTINA – SALTA 4400 | 2024`

**by Analía Elizabeth Gomory**

* ae.gomory@gmail.com

* http://www.linkedin.com/in/elizabeth-gomory-9240b1217

* https://ae-gomory.itch.io

* https://github.com/ElizabethGomory

* https://www.credly.com/users/analia-gomory

* https://orcid.org/0009-0003-0498-7658

**Importación de Librerías y Archivos CSV**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
%matplotlib inline

In [ ]:
test=pd.read_csv('API2_test.csv')
train=pd.read_csv('API2_train.csv')

### A. Descripción de cantidades faltantes para cada variable de las bases de datos.

####**Test**





In [ ]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [ ]:
test.isnull().sum()

,0
PassengerId,0
Pclass,0
Name,0
Sex,0
Age,86
SibSp,0
Parch,0
Ticket,0
Fare,1
Cabin,327


In [ ]:
test.shape

(418, 11)

####**Train**

In [ ]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
train.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [ ]:
train.shape

(891, 12)

In [ ]:
#El resultado de este código se tendrá en cuenta el en punto "C"
train.Cabin.isnull().sum()*100/train.shape[0]
train.drop('Cabin', axis=1, inplace = True)

### B. Completar valores faltantes de las bases de datos.

####**Test**

In [ ]:
amean=test['Age'].mean()
test['Age'] = test['Age'].fillna(amean)
bmean=test['Fare'].mean()
test['Fare'] = test['Fare'].fillna(bmean)

test['Cabin'] = test['Cabin'].fillna(0)
test['Cabin'] = pd.to_numeric(test['Cabin'], errors='coerce')
cmean=test['Cabin'].mean()
test['Cabin'] = test['Cabin'].fillna(cmean)

In [ ]:
test.isnull().sum()

,0
PassengerId,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0
Cabin,0


####**Train**

In [ ]:
dmean = train['Age'].mean()
train['Age'] = train['Age'].fillna(dmean)

train['Embarked'] = train['Embarked'].fillna(0)
train['Embarked'] = pd.to_numeric(train['Embarked'], errors='coerce')
emean = train['Embarked'].mean()
train['Embarked'] =  train['Embarked'].fillna(emean)


In [ ]:
train.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


###C. 1º Modelo de Regresión Logística

####**Train**

Para comenzar el modelo de regresión, se necesitan datos lo más precisos posibles, por lo que se decidió eliminar la columna de "Cabin" ya que, si bien se cumplió con el pedido en el punto "B", éste tiene un porcentaje muy alto de valores nulos: 77%; lo que no permitiría hacer un modelo certero.
En segundo lugar, se eligió esta base de datos ya que cuenta con la columna "Survived", el cual, nos permitirá lograr el objetivo.

In [ ]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,0.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,0.0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,0.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,0.0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,0.0


In [ ]:
#Verificación de los tipos de cada columna
train.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [ ]:
#Modificación de los tipos para poder realizar el modelo
pre_columns = ['Sex', 'Embarked']
for c in pre_columns:
    encoder = LabelEncoder()
    train[c] = encoder.fit_transform(train[c].astype('str'))
    train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    int64  
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    float64
dtypes: float64(3), int64(6), object(2)
memory usage: 76.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass  

In [ ]:
#Imputación de variables
x_columns = ['Age', 'Embarked']
imputer = KNNImputer (n_neighbors=3, weights='uniform')
train[x_columns] = imputer.fit_transform(train[x_columns])

display(train[x_columns].info ())
display(train[x_columns].sample (5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Age       891 non-null    float64
 1   Embarked  891 non-null    float64
dtypes: float64(2)
memory usage: 14.0 KB


None

,Age,Embarked
84,17.000000,0.0
588,22.000000,0.0
629,29.699118,0.0
821,27.000000,0.0
652,21.000000,0.0


####Modelo de Regresión

In [ ]:
#Asignación de Variables para el Modelo de Regresión
columnas = ['Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
nrandom=28
label_column = 'Survived'
x, y = train [columnas], train[label_column]
print(x.columns)
print(y)

Index(['Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')
0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64


In [ ]:
#Entrenamiento de la información
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.25, random_state=nrandom)

In [ ]:
regressor=LogisticRegression(random_state=nrandom)
regressor.fit(x_train, y_train)

LogisticRegression(random_state=28)

In [56]:
# Indicadores de Efectividad

y_pred = regressor.predict(x_val)
acc = accuracy_score(y_val, y_pred)
prec = precision_score(y_val, y_pred)
rec = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
print(f'accuracy : {acc:.4f}')
print(f'precision : {acc:.4f}')
print(f'recall : {acc:.4f}')
print(f'F1 Score : {acc:.4f}')

accuracy : 0.7309
precision : 0.7309
recall : 0.7309
F1 Score : 0.7309


###D. Entrenamiento y determinación del nivel de Accuracy

####Modelado de Árbol de Decisión

In [ ]:
arbol = DecisionTreeClassifier(random_state=nrandom)
arbol.fit(x_train, y_train)

DecisionTreeClassifier(random_state=28)

In [ ]:
y_pred = arbol.predict(x_val)
acc = accuracy_score(y_val, y_pred)
prec = precision_score(y_val, y_pred)
rec = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
print(f'accuracy : {acc:.4f}')
print(f'precisión : {acc:.4f}')
print(f'recall : {acc:.4f}')
print(f'F1 Score : {acc:.4f}')

accuracy : 0.6816
precisión : 0.6816
recall : 0.6816
F1 Score : 0.6816


**Conclusiones**
Tanto en la Regresión Logística como en el Árbol de Decisión, las métricas de *"accuracy, precision, recally f1score"* son casi iguales entre sí. Esto sugiere que los modelos están teniendo un comportamiento similar en términos de predicciones positivas y negativas en el conjunto de validación. Es probable que el conjunto de datos sea equilibrado, es decir: donde la proporción de Sobrevivientes y No Sobrevivientes está casi equilibrada.
La diferencia entre ambos modelos es del 73% contra el 68% lo que nos indica que el modelo de Regresión Logística es más adecuado para este conjunto de datos, lo que indica que el problema de clasificación es más lineal, frente al modelo de Árbol de decisión que es más útil para relacione no lineales. Podríamos concluir que Ambos modelos se encuentran equilibrados en términos de precisión y cobertura, pero con un rendimiento del 68-73%
